# S11 · De las fotografías a evidencia verificable

**Universidad Central · Big Data · 60 minutos de demostración + ampliación autónoma en el mismo cuaderno**

Laura investiga **30 actas SECOP**, **62 fotografías de dos conjuntos** y **30 voces humanas MINDS-14**. Para explicar cada herramienta empezamos con una muestra pequeña y después ampliamos a **122 fuentes**, todo aquí, sin cambiar de cuaderno. Queremos responder qué se lee, qué se observa y qué se escucha, sin confundir herramientas ni inventar relaciones entre fuentes.

1. ¿Cuál es el TOTAL, el DESCUENTO y el NETO A PAGAR que aparecen en un acta?
2. ¿Qué letras aparecen en los anuncios de una fotografía?
3. ¿Qué fotografía se recupera con la consulta «viviendas en una ladera»?
4. ¿Qué dijo una voz humana y cómo lo comprobamos contra el audio?
5. ¿Cómo preservamos archivo, modalidad y huella SHA-256?

**Lo importante es aprender a escribir código**, no ejecutar un programa terminado. **Este es el único cuaderno de la sesión:** FASE 1 (fundamentos) y FASE 2 (análisis por lotes). Cada celda introduce pocas líneas; debes editar variables, probar otras imágenes y explicar un resultado o error. Puedes borrar una celda y reescribirla.

**Fuentes separadas:** las fotos periodísticas no prueban nada sobre las actas; la voz MINDS-14 es bancaria y NO pertenece a las noticias. Las actas pueden contener datos personales; evita publicar OCR completo de nombres o números de cuenta.

## 1 · Entorno y archivos (0–6 min)

**Qué es cada herramienta:** Python ejecuta nuestras instrucciones; Pillow lee píxeles; Path representa rutas de archivos; Tesseract (instalado en Linux) reconoce caracteres. La instalación manual se explica en `INSTALACION_UBUNTU.md`. En Codespaces ya está el programa y sus paquetes, pero debes comprobarlos.

**En la terminal:** `python --version` y `tesseract --list-langs`. Debe aparecer `spa`. Luego escribe estas líneas en una celda vacía.

**Tu cambio:** sustituye la foto `006` por `009` y observa la diferencia.

In [ ]:
from pathlib import Path
from PIL import Image
from IPython.display import display, Audio
import hashlib, io, json, re

RAIZ = Path.cwd()
FOTOS = RAIZ / "datos/raw/fotos"
ACTAS = RAIZ / "datos/raw/actas"
SALIDAS = RAIZ / "outputs"
SALIDAS.mkdir(exist_ok=True)

def sha256(ruta):
    return hashlib.sha256(Path(ruta).read_bytes()).hexdigest()

foto = FOTOS / "eltiempo_006.webp"
imagen = Image.open(foto)
print("Formato:", imagen.format, "Píxeles:", imagen.size)
display(imagen)

## 2 · OCR documental real: un acta de inicio SECOP (6–18 min)

**Pregunta de investigación:** ¿cuál es el valor contractual, cuánto dura y cómo se pagará? Trabajamos con el documento original **secop_acta_728704921.pdf** (consulta sus páginas en el visor).

**OCR** significa reconocimiento óptico de caracteres: Tesseract intenta convertir **píxeles** en palabras. **pytesseract** permite llamarlo desde Python. Un documento con tablas puede confundir el orden de lectura: una frase de una columna puede mezclarse con la etiqueta de otra.

**PSM** (*Page Segmentation Mode*, modo de segmentación de página) indica cómo buscar bloques de texto: **3** detecta el diseño automáticamente (punto de partida para esta página), **6** supone un bloque de texto y **11** intenta encontrar fragmentos dispersos. **No es un porcentaje ni una puntuación de calidad**.

**En pantalla:** muestra el PDF original, selecciona su primera página, ubica los campos VALOR DEL CONTRATO, DURACIÓN y FORMA DE PAGO; después compara con los resultados del OCR completo en el visor.

**Privacidad:** las actas pueden incluir nombres e identificaciones. Para grabar usa un recorte que destaque campos contractuales y no publiques la transcripción completa sin revisarla.

In [ ]:
import pytesseract

# El PDF original permanece intacto; para este ejemplo usamos su primera página PNG.
acta_pdf = RAIZ / "datos/raw/actas_pdf/secop_acta_728704921.pdf"
acta = ACTAS / "secop_acta_728704921_pagina1.png"
pagina = Image.open(acta)
print("Archivo:", acta_pdf.name, "| Imagen:", pagina.size)

# Acercamiento didáctico a las filas de valor y forma de pago.
recorte = pagina.crop((int(pagina.width*0.12), int(pagina.height*0.39),
                       int(pagina.width*0.91), int(pagina.height*0.64)))
display(recorte.resize((780, int(recorte.height*780/recorte.width))))

ocr_3 = pytesseract.image_to_string(pagina, lang="spa", config="--psm 3")
print("Caracteres OCR:", len(ocr_3), "| Líneas:", len(ocr_3.splitlines()))
print("Revisa el texto completo con privacidad en la pestaña OCR del visor.")
for linea in ocr_3.splitlines():
    if any(campo in linea.upper() for campo in
           ("VALOR DEL CONTRATO", "FORMA DE PAGO", "DURACI", "ACTA DE INICIO")):
        print("Candidato:", linea.strip()[:160])

### Verificación independiente: no confundir salida OCR y evidencia visual

En la **primera página original** se observan, mediante lectura humana:

| Campo | Referencia visual del PDF |
|---|---|
| Tipo | Acta de inicio |
| Contrato | 006-2026 |
| Valor contractual | **$14.760.000 COP** |
| Plazo | **6 meses** |
| Forma de pago | **6 pagos de $2.460.000 COP** |

**Importante:** estos valores son la **referencia visual para cotejo**, **NO** una supuesta predicción del motor. Puede ocurrir que el OCR encuentre palabras de una columna y las ubique incorrectamente junto a otra etiqueta.

**Comprobación aritmética independiente:** 6 × 2.460.000 = 14.760.000. Esto verifica coherencia matemática entre cifras observadas, **no certifica autenticidad de la contratación**.

**Reto:** observa y explica si Tesseract reconoció ambas cantidades, si detectó el número de contrato y si confundió letras o columnas. Prueba PSM 3, 6 y 11 sobre **el mismo archivo**.

In [ ]:
# Variamos solo el PSM; el documento y el idioma siguen siendo iguales.
modos = [3, 6, 11]
for psm in modos:
    texto_psm = pytesseract.image_to_string(
        pagina, lang="spa", config=f"--psm {psm}")
    lineas_utiles = [linea.strip()[:140] for linea in texto_psm.splitlines()
                     if any(x in linea.upper() for x in
                            ("VALOR DEL CONTRATO", "FORMA DE PAGO", "DURACI"))]
    print(f"PSM {psm}: {len(texto_psm)} caracteres · {len(lineas_utiles)} líneas candidatas")
    print(lineas_utiles[:3] or ["No se recuperó ninguna etiqueta buscada"])

# Referencia OBSERVADA EN LA FUENTE, no predicción de OCR:
valor_visual = 14_760_000
cuotas_visual = 6
pago_visual = 2_460_000
print("¿Coincide el producto?", cuotas_visual * pago_visual == valor_visual)

### Extraer campos: primero encontrar evidencia, después interpretar

Para extraer un valor no basta encontrar cualquier número. Debemos localizar la etiqueta **VALOR DEL CONTRATO** y examinar el texto cercano. Escribe una función corta que devuelva un **fragmento de OCR sin corregir**; si no reconoce la etiqueta, reporta que no la encontró. **No sustituyas silenciosamente un texto fallido por los valores de la referencia visual.**

In [ ]:
def fragmento_cercano(texto, etiqueta, caracteres=220):
    coincidencia = re.search(etiqueta, texto, flags=re.IGNORECASE)
    if not coincidencia:
        return None
    return texto[coincidencia.start():coincidencia.start()+caracteres]

for campo in ("VALOR DEL CONTRATO", "FORMA DE PAGO", "DURACI"):
    fragmento = fragmento_cercano(ocr_3, campo)
    print(campo, "→", "[No detectado]" if fragmento is None else fragmento[:160].replace("\n", " "))

# Cambia caracteres=220 a 400 y comprueba si se mezclan columnas.
# En el visor puedes leer el OCR completo y comparar con el PDF original.

## 3 · OCR en fotografías: por qué un logotipo puede fallar

Una persona lee en la fotografía 023 «Lotería del VALLE · Cambia tu vida», pero Tesseract puede devolver letras incompletas. ¿Por qué?

**La imagen es un diseño publicitario**, no un documento de texto continuo: las letras son curvas, de colores, tienen contornos decorativos y conviven con una persona que ocupa casi media fotografía. El OCR no sabe automáticamente qué región queremos analizar.

**Experimento paso a paso:**
1. Abre en el visor `eltiempo_023.webp` y obsérvala.
2. Ejecuta el bucle sobre cuatro fotografías con **PSM_FOTOS = 11**.
3. **EDITA AQUÍ:** cambia PSM_FOTOS a 6 y vuelve a ejecutar.
4. Examina el **texto completo**. No uses `repr(texto[:130])`, que recorta la información y muestra barras como `\n` en vez de los saltos reales.
5. Después recorta únicamente la palabra VALLE y compara el PSM 7.

**Pregunta pedagógica:** ¿por qué una función puede terminar sin errores y, aun así, producir un resultado OCR incorrecto?

In [ ]:
nombres = ["eltiempo_006.webp", "eltiempo_023.webp",
           "eltiempo_001.jpg", "eltiempo_009.webp"]
PSM_FOTOS = 11  # EDITA AQUÍ: vuelve a ejecutar con 6
textos_fotos = {}  # Diccionario para conservar cada resultado real

for nombre in nombres:
    ruta = FOTOS / nombre
    imagen_foto = Image.open(ruta)
    leido = pytesseract.image_to_string(
        imagen_foto, lang="spa", config=f"--psm {PSM_FOTOS}"
    ).strip()
    textos_fotos[nombre] = leido
    print(f"\n{'='*15} {nombre} {'='*15}")
    print("Caracteres:", len(leido))
    print(leido if leido else "(Tesseract no reconoció palabras)")

### Laboratorio guiado · Aislar la palabra VALLE

**¿Qué debes mostrar?** La fotografía completa y, al lado, el recorte que contiene las letras de colores. La comparación evita confundir la salida incorrecta de OCR con una descripción de la imagen.

**¿Qué significan las instrucciones?**
- `Image.open` lee la foto y `size` proporciona ancho y alto.
- `crop` necesita cuatro coordenadas (izquierda, arriba, derecha, abajo).
- `PSM 7` indica a Tesseract que el recorte tiene **una sola línea**. `PSM 6` supone un bloque uniforme y `PSM 11` busca texto disperso.
- `print` enseña **la respuesta real del motor**, sin reemplazarla por la palabra esperada.

**EDITA AQUÍ:** cambia `ZONA_VALLE` y después `MODOS_LOGO`. El primer recorte es una propuesta para esta foto; prueba otros y explica cuál fue mejor.

**Referencia humana:** la palabra que vemos es VALLE. Si OCR devuelve «VALE», «(VALLE)» o algo incompleto, eso se registra como predicción, no como verdad.

In [ ]:
ruta_logo = FOTOS / "eltiempo_023.webp"  # EDITA AQUÍ: foto fuente
foto_logo = Image.open(ruta_logo).convert("RGB")
ancho, alto = foto_logo.size
print("Dimensiones:", ancho, "×", alto)

# EDITA AQUÍ: cuatro límites proporcionales; 0 es inicio y 1 es extremo.
ZONA_VALLE = (175/1200, 240/675, 540/1200, 375/675)
caja = (round(ancho*ZONA_VALLE[0]), round(alto*ZONA_VALLE[1]),
        round(ancho*ZONA_VALLE[2]), round(alto*ZONA_VALLE[3]))
region_valle = foto_logo.crop(caja)
display(foto_logo)
display(region_valle)

MODOS_LOGO = [6, 7, 11]  # EDITA AQUÍ: compara PSM
for psm_logo in MODOS_LOGO:
    salida = pytesseract.image_to_string(
        region_valle, lang="spa", config=f"--psm {psm_logo}"
    ).strip()
    print(f"PSM {psm_logo}:", salida or "(no reconoció texto)")

### Experimento controlado · ¿Ampliar la imagen mejora OCR?

**¿Por qué un aumento podría ayudar?** Las letras tienen pocos píxeles y contornos curvos. `resize` con `LANCZOS` interpola nuevos píxeles y puede facilitar la segmentación, aunque **no recupera información ausente**.

Primero mantén el recorte y PSM 7 constantes, y cambia **solo FACTOR = 1 → 2**. Después mantén la escala y prueba **USAR_CONTRASTE = True**.

**¿Qué hace cada función?** `ImageOps.autocontrast` redistribuye intensidades; `ImageFilter.UnsharpMask` resalta bordes; `image_to_string` devuelve la predicción OCR. Estas transformaciones pueden mejorar o empeorar resultados, así que necesitamos comparar, no suponer.

**EDITA AQUÍ:** FACTOR, USAR_CONTRASTE y PSM_RECORTADO.

In [ ]:
from PIL import ImageOps, ImageFilter

FACTOR = 2             # EDITA AQUÍ: probar 1, 2 o 3
USAR_CONTRASTE = False # EDITA AQUÍ: probar después True
PSM_RECORTADO = 7      # Una sola línea

region_procesada = region_valle.resize(
    (region_valle.width*FACTOR, region_valle.height*FACTOR),
    Image.Resampling.LANCZOS
)
if USAR_CONTRASTE:
    region_procesada = ImageOps.autocontrast(
        ImageOps.grayscale(region_procesada))
    region_procesada = region_procesada.filter(
        ImageFilter.UnsharpMask(radius=1, percent=130, threshold=2))
display(region_procesada)

salida_real = pytesseract.image_to_string(
    region_procesada, lang="spa", config=f"--psm {PSM_RECORTADO}"
).strip()
print("OCR real:", salida_real or "(no detectó texto)")
print("Referencia humana: VALLE (no es predicción OCR)")

### Evaluar la salida: texto, posición y confianza interna

`image_to_data` devuelve palabras con sus coordenadas y una confianza estimada por Tesseract. Esto muestra **qué parte del recorte interpretó como texto**.

Una puntuación interna de 90 **no es un 90 % de exactitud real**. Para medir exactitud necesitamos una referencia humana independiente y una métrica de error.

**EDITA AQUÍ:** cambia CONFIANZA_MINIMA de 0 a 40, comprueba qué palabras se omiten y mira si eran correctas en la imagen.

In [ ]:
datos_ocr = pytesseract.image_to_data(
    region_procesada, lang="spa", config=f"--psm {PSM_RECORTADO}",
    output_type=pytesseract.Output.DICT)
CONFIANZA_MINIMA = 0  # EDITA AQUÍ: prueba 40
for i, palabra in enumerate(datos_ocr["text"]):
    if not palabra.strip():
        continue
    confianza = float(datos_ocr["conf"][i])
    if confianza >= CONFIANZA_MINIMA:
        print("Palabra:", palabra, "| confianza interna:", round(confianza, 1),
              "| posición:", (datos_ocr["left"][i], datos_ocr["top"][i]))

## 4 · Búsqueda por significado con EmbeddingGemma 2 (25–35 min)

**Pregunta:** ¿cuál fotografía muestra viviendas en una ladera? OCR reconoce letras; un **embedding** representa contenido en un vector de **768 números**.

**Herramientas:** Ollama ejecuta `embeddinggemma-2:440m` en un servidor local. La API `POST /api/embed` recibe texto o el objeto `{"image": "<bytes base64>"}`. **Base64** transporta bytes dentro de JSON: no es inteligencia artificial. Pillow abre y normaliza WebP/JPG a JPEG RGB para evitar diferencias de formato sin alterar el archivo fuente.

**Preparación del Codespace:** ejecuta en Terminal `bash preparar_embeddinggemma.sh` y comprueba el listado con `ollama list`.

**Lo que haces en el código siguiente:**
1. Abres una foto con Pillow, corriges orientación EXIF, la conviertes a RGB y la limitas a 768 píxeles de lado.
2. La guardas como JPEG temporal **en memoria**, la codificas a Base64 y construyes el JSON.
3. Compruebas **una sola imagen** antes de empezar las otras. Imprimes el nombre de archivo.
4. Ante HTTP 400, lees el **mensaje real del servidor**, no solo `raise_for_status()`.
5. Verificas que el resultado tenga 768 dimensiones.

**EDITA AQUÍ:** `MAX_LADO = 768` → 512 y `primer_nombre` para cambiar imagen. Observa tamaño Base64, tiempo y resultado. La búsqueda por similitud no certifica hechos.

In [ ]:
import requests, base64, math
from PIL import ImageOps
from io import BytesIO

MODELO = "embeddinggemma-2:440m"
API = "http://127.0.0.1:11434/api/embed"
MAX_LADO = 768  # EDITA AQUÍ: prueba con 512

def preparar_imagen_base64(ruta):
    """Convierte WebP/JPG a JPEG RGB en memoria, conservando el original."""
    with Image.open(ruta) as original:
        imagen = ImageOps.exif_transpose(original).convert("RGB")
        imagen.thumbnail((MAX_LADO, MAX_LADO))
        memoria = BytesIO()
        imagen.save(memoria, format="JPEG", quality=85)
    return base64.b64encode(memoria.getvalue()).decode("ascii")

def vector(entrada, nombre="consulta"):
    """Enviar la petición y mostrar el error REAL que devuelve Ollama."""
    try:
        respuesta = requests.post(API, json={"model": MODELO, "input": entrada},
                                  timeout=240)
    except requests.RequestException as exc:
        raise RuntimeError("Ollama no responde. Prueba bash iniciar_ollama.sh") from exc
    if not respuesta.ok:
        raise RuntimeError(f"Ollama rechazó {nombre} (HTTP {respuesta.status_code}): "
                           f"{respuesta.text[:1500]}")
    vectores = respuesta.json().get("embeddings", [])
    if len(vectores) != 1 or len(vectores[0]) != 768:
        raise RuntimeError(f"No se obtuvo un vector de 768 dimensiones para {nombre}")
    return vectores[0]

# PREVIO: verificamos una fotografía antes del bucle.
primer_nombre = nombres[0]  # EDITA AQUÍ: también prueba eltiempo_023.webp
contenido = preparar_imagen_base64(FOTOS / primer_nombre)
print("Primera foto:", primer_nombre, "longitud Base64:", len(contenido))
primer_vector = vector({"image": contenido}, primer_nombre)
print("Comprobación correcta:", len(primer_vector), "dimensiones")

vectores_fotos = {primer_nombre: primer_vector}
for nombre in nombres[1:]:
    print("Procesando", nombre)
    vectores_fotos[nombre] = vector(
        {"image": preparar_imagen_base64(FOTOS / nombre)}, nombre
    )
    print("Vector obtenido:", len(vectores_fotos[nombre]), "valores")
assert len(vectores_fotos) == 4

### Tu búsqueda: cambia la pregunta

**Conceptos:** producto punto y norma de un vector. El coseno divide el producto punto entre las normas. Este cálculo es pequeño y comprensible; no hace falta una base de datos vectorial para cuatro imágenes.

Prueba: «viviendas en una ladera», «medicamentos», «taquillas de transporte» y «sorteo de lotería». **Evalúa manualmente** si el primer resultado corresponde a la imagen, sin afirmar precisión universal.

In [ ]:
def coseno(a, b):
    producto = sum(x*y for x, y in zip(a, b))
    norma_a = math.sqrt(sum(x*x for x in a))
    norma_b = math.sqrt(sum(y*y for y in b))
    return producto / (norma_a * norma_b)

pregunta = "viviendas en una ladera"  # TU TURNO: modifica la consulta
consulta = vector("task: search result | query: " + pregunta)
ranking = sorted(((nombre, coseno(consulta, valores))
                  for nombre, valores in vectores_fotos.items()),
                 key=lambda fila: -fila[1])
for nombre, similitud in ranking:
    print(f"{similitud:.3f} · {nombre}")
display(Image.open(FOTOS / ranking[0][0]))

## 5 · Transcribir voz REAL con Faster-Whisper (35–44 min)

**Fuente autorizada:** PolyAI MINDS-14, es-ES, licencia CC BY 4.0. Son conversaciones bancarias, **NO audios de las fotos de El Tiempo**.

Esta celda descarga el **Parquet** de fuente (~39 MB) desde Hugging Face, verifica la huella SHA-256 del corpus y conserva el WAV original de una persona. El audio puede contener el códec G.711: **WAV es un contenedor, no un códec**.

Antes de hablar del ASR, **escucha** el WAV. La transcripción `referencia` pertenece al corpus, no es la predicción de Whisper. Si falla la descarga o el hash, detente: no fabriques datos de sustitución.

In [ ]:
from urllib.request import Request, urlopen
import pandas as pd

URL = ("https://huggingface.co/datasets/PolyAI/minds14/resolve/main/"
       "es-ES/train-00000-of-00001.parquet")
SHA_CORPUS = "989bf2f90676a5eb5852fe785a03b1d9b97763e0febbd0fd0a3f489afd83badf"
corpus = RAIZ / "datos/minds14_es-ES.parquet"

if not corpus.exists():
    peticion = Request(URL, headers={"User-Agent": "BigData-UCentral-educativo/1.0"})
    with urlopen(peticion, timeout=120) as red, corpus.open("wb") as salida:
        while bloque := red.read(1024 * 1024):
            salida.write(bloque)
            if salida.tell() > 100_000_000:
                raise ValueError("Archivo supera los 100 MB")
if sha256(corpus) != SHA_CORPUS:
    raise ValueError("El Parquet de audio no coincide con la fuente auditada")

audios = pd.read_parquet(corpus, columns=["audio", "path", "transcription"])
fila = audios.sort_values("path").iloc[0]
wav = RAIZ / "datos/raw/voz_minds14.wav"
wav.write_bytes(fila["audio"]["bytes"])
referencia = str(fila["transcription"])
print("Fuente:", fila["path"], "| SHA:", sha256(wav)[:16])
display(Audio(filename=str(wav)))

### ¿Qué aprende el modelo Whisper?

**Faster-Whisper** ejecuta `Whisper tiny` en CPU con cuantización `int8`: representación de 8 bits para reducir uso de memoria. El modelo recibe la grabación, no copia la transcripción humana. Devuelve segmentos con **inicio, fin y texto**.

**Tu verificación:** identifica una frase problemática, reproduce el intervalo y compara lo que dijo la persona con lo que predijo el modelo. No ocultes errores.

In [ ]:
from faster_whisper import WhisperModel

modelo_voz = WhisperModel("tiny", device="cpu", compute_type="int8",
                          cpu_threads=2)
segmentos, info = modelo_voz.transcribe(str(wav), language="es", beam_size=2)
segmentos = list(segmentos)
hipotesis = " ".join(s.text.strip() for s in segmentos)
print("Predicción Whisper:", hipotesis)
print("Referencia humana:", referencia)
for s in segmentos:
    print(round(s.start, 1), "→", round(s.end, 1), s.text)

### Medir WER con una función pequeña

**WER:** tasa de error de palabras = número mínimo de inserciones, borrados y sustituciones dividido entre las palabras de referencia. Aquí la referencia humana existe porque viene del corpus. Sin ella **NO** afirmes una tasa de error. El WER solo describe este audio y este método de comparación.

In [ ]:
def palabras(texto):
    return re.findall(r"\w+", texto.lower())

def errores(a, b):
    anterior = list(range(len(b) + 1))
    for i, palabra in enumerate(a, 1):
        actual = [i]
        for j, otra in enumerate(b, 1):
            actual.append(min(actual[-1] + 1, anterior[j] + 1,
                              anterior[j-1] + (palabra != otra)))
        anterior = actual
    return anterior[-1]

ref = palabras(referencia)
pred = palabras(hipotesis)
print("WER de este audio:", round(errores(ref, pred) / len(ref), 3))

## 6 · ETL con Python escrito por ti (44–54 min)

**E = extraer:** abrir original, recuperar texto o voz.  
**T = transformar:** crear diccionarios comparables, conservar tipo y procedencia.  
**L = cargar:** guardar **JSONL** (un objeto por línea) y **Parquet** (columnas tipadas).

No vamos a ejecutar un motor de 400 líneas. Vas a escribir `for`, `dict`, `json.dumps` y `to_parquet`.

**Privacidad:** **no publiques el OCR completo de las actas**: pueden contener identificaciones y cuentas. Guarda solo la referencia al archivo y los metadatos mínimos. Una huella SHA-256 prueba igualdad de bytes, no que la fuente diga la verdad.

In [ ]:
registros = []
for nombre in nombres:
    archivo = FOTOS / nombre
    registros.append({"archivo": str(archivo.relative_to(RAIZ)),
                      "modalidad": "foto_periodistica",
                      "sha256": sha256(archivo),
                      "texto_recortado": textos_fotos[nombre][:100],
                      "credito": "ver ficha editorial"})

registros.append({"archivo": str(acta.relative_to(RAIZ)),
                  "modalidad": "documento_secop", "sha256": sha256(acta),
                  "texto_recortado": "[OCR protegido: datos personales]",
                  "credito": "documento público SECOP, revisar contexto"})
registros.append({"archivo": str(wav.relative_to(RAIZ)),
                  "modalidad": "audio_minds14", "sha256": sha256(wav),
                  "texto_recortado": hipotesis[:100],
                  "credito": "PolyAI MINDS-14 es-ES, CC BY 4.0"})

ruta_jsonl = SALIDAS / "catalogo.jsonl"
with ruta_jsonl.open("w", encoding="utf-8") as salida:
    for dato in registros:
        salida.write(json.dumps(dato, ensure_ascii=False) + "\n")
pd.DataFrame(registros).to_parquet(SALIDAS / "catalogo.parquet", index=False)
display(pd.DataFrame(registros)[["archivo", "modalidad", "sha256"]])

### Verificar datos sin confiar en una salida bonita

**Comprueba:** reabrir JSONL y Parquet; contar filas; comparar los hashes actuales con los que guardaste. Una fuente alterada debe detectarse. Si quieres experimentar, modifica **una copia**, nunca el archivo RAW.

In [ ]:
filas = [json.loads(linea) for linea in ruta_jsonl.read_text().splitlines()]
tabla = pd.read_parquet(SALIDAS / "catalogo.parquet")
assert len(filas) == len(tabla)
for fila in filas:
    assert sha256(RAIZ / fila["archivo"]) == fila["sha256"]
print("OK:", len(filas), "registros, integridad de originales comprobada")

## Transición · De lo aprendido a 122 fuentes

Acabas de crear las piezas esenciales. Ahora **no cambies de cuaderno**: la siguiente sección aplica esas mismas ideas a lotes. Puedes modificar al menos tres líneas de Python:

- **Documento:** cambia a la segunda acta y compara PSM 6/11; lee visualmente un valor.
- **Imagen:** cambia pregunta y compara los cuatro resultados de búsqueda; verifica el original.
- **Audio:** toma otro registro autorizado del corpus y vuelve a transcribir, comparando la referencia humana.

**Producto breve:** `pregunta | archivo | herramienta | resultado | cómo lo verifiqué | posible error`.

**Criterio de aprendizaje:** debes poder responder **qué hace cada línea esencial**, por qué OCR no describe escenas, por qué el vector no certifica hechos, cómo se comprueba WER y cómo adaptar el flujo a otro conjunto propio. La mera ejecución de una celda no acredita aprendizaje.

**Referencias:** Tesseract OCR; Faster-Whisper (SYSTRAN); EmbeddingGemma 2 (Google/Ollama); PolyAI MINDS-14 es-ES, CC BY 4.0. Fotos: El Tiempo, créditos pendientes de curaduría por imagen. SECOP: documentos públicos, evita difundir datos personales.

---

# FASE 2 · Las mismas funciones, ahora sobre 122 fuentes

**No abras otro cuaderno.** Este bloque sigue al anterior: primero **aprendiste cada motor con un ejemplo**, ahora **escribes bucles de Python** y cambias parámetros para procesar un corpus mayor.

- **62 fotografías** = 30 de `datos/raw/fotos` + 32 de `datos/raw/fotos_catalogadas`.
- **30 actas SECOP** = 30 PDF originales + 30 PNG derivados de su primera página; se cuentan **30 documentos**, no 60.
- **30 voces humanas** = descargas autorizadas PolyAI MINDS-14 (CC BY 4.0), convertidas a PCM16 sin reemplazar originales.
- **122 fuentes** = 62 fotos + 30 actas + 30 voces. No implica 122 noticias ni vínculos entre fuentes.

Durante la **grabación de 60 minutos** muestra los resultados de muestras pequeñas; fuera del video cambia `N_ACTAS=5 → 30`, `N_FOTOS=8 → 62`, `N_VISION=8 → 62` y `N_ASR=3 → 30`. Las descargas e inferencias CPU sobre todo el lote pueden tardar y no necesitan bloquear la narración.

**Para inspeccionar cada fuente:** abre la pestaña del visor del Codespace en el puerto privado 8501. Allí puedes avanzar y retroceder entre archivos, ver el PDF entero junto a su PNG, abrir fotos y escuchar WAV.


## 1 · Construir el inventario

Aprende a recorrer carpetas con Path y a diferenciar originales de derivados. Los 30 PNG son vistas de los PDF: no cuentan como 30 actas adicionales. Calcula la huella SHA-256 y compara registros distintos por nombre con archivos idénticos por bytes.

In [ ]:
from pathlib import Path
from PIL import Image
from IPython.display import display, Audio
import pandas as pd
import hashlib, json, re, time, io, math

BASE = Path.cwd()
FOTOS = sorted((BASE/"datos/raw/fotos").glob("eltiempo_*"))
FOTOS_FICHA = sorted((BASE/"datos/raw/fotos_catalogadas").glob("*.webp"))
ACTAS = sorted((BASE/"datos/raw/actas_pdf").glob("*.pdf"))
PAGINAS = BASE/"datos/raw/actas"
OUT = BASE/"outputs"
OUT.mkdir(exist_ok=True)

def sha256(ruta):
    h = hashlib.sha256()
    with Path(ruta).open("rb") as f:
        for bloque in iter(lambda: f.read(1024*1024), b""):
            h.update(bloque)
    return h.hexdigest()

print("Fotos:", len(FOTOS), "+", len(FOTOS_FICHA))
print("Actas:", len(ACTAS), "PDF y", len(list(PAGINAS.glob("*.png"))), "PNG")
assert (len(FOTOS), len(FOTOS_FICHA), len(ACTAS)) == (30, 32, 30)
fuentes = FOTOS + FOTOS_FICHA + ACTAS
inv = pd.DataFrame([{"ruta":str(p.relative_to(BASE)), "sha256":sha256(p),
                     "mb":round(p.stat().st_size/1e6,3)} for p in fuentes])
print("Registros:",len(inv),"Huellas distintas:",inv.sha256.nunique())
display(inv.head())

## 2 · OCR documental sobre 5 o 30 actas

Tesseract reconoce caracteres de la primera página PNG; conservamos el PDF completo como documento fuente. Mide tiempo por documento y si se reconocieron palabras como TOTAL o PAGAR. **Detectar una palabra no es medir exactitud**.

En el video usa N_ACTAS=5; para la ampliación usa 30. No compartas OCR completo de documentos con identificaciones.

In [ ]:
import pytesseract

N_ACTAS = 5  # Cambiar a 30
filas_actas = []
for pdf in ACTAS[:N_ACTAS]:
    pagina = PAGINAS/(pdf.stem+"_pagina1.png")
    inicio = time.perf_counter()
    texto = pytesseract.image_to_string(Image.open(pagina),lang="spa",config="--psm 6")
    filas_actas.append({
        "pdf":pdf.name,"sha256":sha256(pdf),
        "segundos":round(time.perf_counter()-inicio,2),
        "caracteres":len(texto),"total":"TOTAL" in texto.upper(),
        "pagar":"PAGAR" in texto.upper()
    })
tabla_actas = pd.DataFrame(filas_actas)
display(tabla_actas)
print("Media de segundos:",round(tabla_actas.segundos.mean(),2))

## 3 · OCR sobre 8 o 62 fotografías

Muchas fotos tienen objetos pero no palabras. Aplica OCR y cuenta caracteres; revisa manualmente falsos positivos en el visor. Elige N_FOTOS=8 para la grabación y 62 para comparar todo el lote. La longitud de la salida NO demuestra su calidad.

In [ ]:
TODAS_FOTOS = FOTOS+FOTOS_FICHA
N_FOTOS = 8  # Cambiar a 62
filas_fotos = []
for foto in TODAS_FOTOS[:N_FOTOS]:
    inicio = time.perf_counter()
    leido = pytesseract.image_to_string(Image.open(foto),lang="spa",config="--psm 11")
    filas_fotos.append({"foto":foto.name,"caracteres":len(leido.strip()),
                         "segundos":round(time.perf_counter()-inicio,2)})
tabla_fotos = pd.DataFrame(filas_fotos)
display(tabla_fotos.sort_values("caracteres",ascending=False).head(8))

## 4 · Escalar a 8 y después 62 imágenes sin esconder errores

**Primero comprueba UNA fotografía**, después procesa 8 y finalmente 62. Aunque el modelo admite imagen y texto, un archivo WebP puede estar codificado de forma particular; **normalizamos temporalmente todas las entradas a JPEG RGB** con Pillow, manteniendo sus bytes originales. Esto no cambia las fuentes ni garantiza el éxito de cualquier inferencia.

**Explica y edita:** `N_VISION` define cuántos archivos se procesan; `MAX_LADO` limita los píxeles; `imagen_para_modelo` construye Base64; `embedding` llama a Ollama y **muestra el cuerpo HTTP cuando ocurre un error 400**. `cache` conserva vectores con SHA-256, modelo y parámetros de conversión.

**Tu experimento:** 1) prueba un archivo; 2) amplía a ocho; 3) consulta «viviendas en una ladera»; 4) verifica los primeros resultados abriendo las fotos; 5) cambia a 62 fuera de la clase.

In [ ]:
import requests, base64
from PIL import ImageOps
from io import BytesIO

MODELO = "embeddinggemma-2:440m"
API = "http://127.0.0.1:11434/api/embed"
MAX_LADO = 768  # EDITA AQUÍ: reduce a 512 y compara tiempo/tamaño

def imagen_para_modelo(ruta):
    """Uniforma formatos WebP/JPG a JPEG RGB, sin modificar la foto."""
    with Image.open(ruta) as origen:
        imagen = ImageOps.exif_transpose(origen).convert("RGB")
        imagen.thumbnail((MAX_LADO, MAX_LADO))
        memoria = BytesIO()
        imagen.save(memoria, format="JPEG", quality=85)
    return base64.b64encode(memoria.getvalue()).decode("ascii")

def embedding(entrada, archivo="consulta"):
    """Envía imagen/texto a Ollama y expone sus errores detallados."""
    try:
        r = requests.post(API, json={"model": MODELO, "input": entrada}, timeout=240)
    except requests.RequestException as exc:
        raise RuntimeError("Ollama no responde; verifica bash iniciar_ollama.sh") from exc
    if not r.ok:
        raise RuntimeError(f"Archivo {archivo}, HTTP {r.status_code}: "
                           f"{r.text[:1500]}")
    resultado = r.json().get("embeddings", [])
    if len(resultado) != 1 or len(resultado[0]) != 768:
        raise RuntimeError(f"{archivo}: salida incorrecta; se esperaban 768 valores")
    return resultado[0]

# DIAGNÓSTICO: probar una imagen ANTES de empezar el lote.
primera = TODAS_FOTOS[0]
print("Prueba de imagen:", primera.name)
prueba = embedding({"image": imagen_para_modelo(primera)}, primera.name)
print("Dimensiones verificadas:", len(prueba))

cache = OUT / "vectores_440m.json"
vectores = json.loads(cache.read_text()) if cache.exists() else {}
N_VISION = 8  # EDITA AQUÍ: una vez funciona, cambia a 62

for foto in TODAS_FOTOS[:N_VISION]:
    key = str(foto.relative_to(BASE))
    huella = sha256(foto)
    clave_cache = f"{MODELO}|{MAX_LADO}|JPEG85|{huella}"
    if key not in vectores or vectores[key].get("clave_cache") != clave_cache:
        print("Calculando:", key)
        resultado = embedding({"image": imagen_para_modelo(foto)}, key)
        vectores[key] = {"sha256": huella, "clave_cache": clave_cache,
                        "vector": resultado}
        cache.write_text(json.dumps(vectores), encoding="utf-8")
    else:
        print("Ya calculado:", key)
print("Vectores en caché:", len(vectores))
assert all(len(x["vector"]) == 768 for x in vectores.values())

In [ ]:
def coseno(a,b):
    num = sum(x*y for x,y in zip(a,b))
    norm_a = math.sqrt(sum(x*x for x in a))
    norm_b = math.sqrt(sum(y*y for y in b))
    return num/(norm_a*norm_b)

pregunta = "viviendas en una ladera"  # Cambia a medicamentos o lotería
consulta = embedding("task: search result | query: "+pregunta)
ranking = sorted(((k,coseno(consulta,v["vector"])) for k,v in vectores.items()),
                 key=lambda x:-x[1])
display(pd.DataFrame(ranking[:10],columns=["archivo","similitud"]))
display(Image.open(BASE/ranking[0][0]))

## 5 · Descargar 30 voces humanas con permiso y referencia

Fuente PolyAI MINDS-14 es-ES, licencia CC BY 4.0. Recupera el Parquet real (~39 MB) desde Hugging Face; verifica su SHA-256 antes de leer. Elige una voz de cada una de varias intenciones seleccionadas por rondas. Estos audios bancarios NO son relatos de las fotografías.

In [ ]:
from urllib.request import Request,urlopen
URL = ("https://huggingface.co/datasets/PolyAI/minds14/resolve/main/"
       "es-ES/train-00000-of-00001.parquet")
SHA_ESPERADO = "989bf2f90676a5eb5852fe785a03b1d9b97763e0febbd0fd0a3f489afd83badf"
corpus = BASE/"datos/minds14_es-ES.parquet"
if not corpus.exists():
    req=Request(URL,headers={"User-Agent":"BigData-UCentral-educativo/1.0"})
    with urlopen(req,timeout=180) as web, corpus.open("wb") as out:
        while bloque := web.read(1024*1024):
            out.write(bloque)
            if out.tell()>100_000_000: raise ValueError("Descarga excede 100 MB")
assert sha256(corpus)==SHA_ESPERADO,"Corpus distinto del auditado"
tabla = pd.read_parquet(corpus,columns=["audio","path","transcription","intent_class"])
ordenadas = tabla.sort_values("path", kind="stable").copy()
ordenadas["ronda"] = ordenadas.groupby("intent_class").cumcount()
elegidas = ordenadas.sort_values(["ronda", "intent_class", "path"], kind="stable").head(30)
assert len(elegidas)==30 and elegidas["path"].nunique()==30
print("Treinta grabaciones únicas por rondas de intención")

## 6 · Conservar los originales y convertir a WAV PCM16

Un WAV puede contener códecs distintos. SoundFile decodifica la señal y wave guarda una copia PCM16 mono compatible con el reproductor. Conserva hash de fuente y derivado. Después abre cada archivo en el visor.

In [ ]:
import soundfile as sf
import numpy as np
import wave

RAW_AUDIO=BASE/"datos/raw/audios"
DER_AUDIO=BASE/"datos/derived/audios"
RAW_AUDIO.mkdir(parents=True,exist_ok=True)
DER_AUDIO.mkdir(parents=True,exist_ok=True)
voces=[]
for n,(_,fila) in enumerate(elegidas.iterrows(),1):
    original=bytes(fila["audio"]["bytes"])
    fuente=RAW_AUDIO/f"voz_{n:02}_fuente.wav"
    if not fuente.exists(): fuente.write_bytes(original)
    assert sha256(fuente)==hashlib.sha256(original).hexdigest()
    with sf.SoundFile(io.BytesIO(original)) as dec:
        tasa=dec.samplerate
        samples=dec.read(dtype="float32",always_2d=True)
    mono=samples.mean(axis=1)
    pcm=(np.clip(mono,-1,1)*32767).astype("<i2")
    destino=DER_AUDIO/f"voz_{n:02}.wav"
    with wave.open(str(destino),"wb") as wav:
        wav.setnchannels(1);wav.setsampwidth(2);wav.setframerate(tasa)
        wav.writeframes(pcm.tobytes())
    voces.append({
        "registro":str(fila["path"]),"intencion":int(fila["intent_class"]),
        "referencia":str(fila["transcription"]),
        "fuente":str(fuente.relative_to(BASE)),"pcm16":str(destino.relative_to(BASE)),
        "sha_fuente":sha256(fuente),"sha_pcm16":sha256(destino)
    })
manifest=BASE/"datos/manifiestos/audios_minds14.json"
manifest.parent.mkdir(parents=True,exist_ok=True)
manifest.write_text(json.dumps({"dataset":"PolyAI/minds14","subset":"es-ES",
    "license":"CC BY 4.0","source_url":URL,"sha256_parquet":SHA_ESPERADO,
    "audios":voces},ensure_ascii=False,indent=2),encoding="utf-8")
display(Audio(filename=str(BASE/voces[0]["pcm16"])))
print("Audios disponibles en el visor:",len(voces))

## 7 · Medir la tasa de error de palabras con Whisper

Whisper tiny ejecuta ASR en CPU con cuantización INT8. La transcripción humana de MINDS-14 sirve como referencia del mismo registro. WER = inserciones + borrados + sustituciones / palabras de referencia. Compara errores y tiempo de tres voces en video y diez para el reto.

In [ ]:
from faster_whisper import WhisperModel

whisper=WhisperModel("tiny",device="cpu",compute_type="int8",cpu_threads=2)
def palabras(t): return re.findall(r"\w+",t.lower())
def errores(a,b):
    anterior=list(range(len(b)+1))
    for i,x in enumerate(a,1):
        fila=[i]
        for j,y in enumerate(b,1):
            fila.append(min(fila[-1]+1,anterior[j]+1,anterior[j-1]+(x!=y)))
        anterior=fila
    return anterior[-1]

N_ASR=3  # Cambiar a 30
salidas=[]
for voz in voces[:N_ASR]:
    ruta=BASE/voz["pcm16"]
    inicio=time.perf_counter()
    segmentos,_=whisper.transcribe(str(ruta),language="es",beam_size=2)
    transcripcion=" ".join(s.text.strip() for s in segmentos)
    ref=palabras(voz["referencia"])
    salidas.append({"audio":ruta.name,"wer":round(errores(ref,palabras(transcripcion))/len(ref),3),
                    "segundos":round(time.perf_counter()-inicio,2)})
tabla_wer=pd.DataFrame(salidas)
display(tabla_wer.sort_values("wer",ascending=False))

## 8 · Crear y auditar el catálogo de 122 fuentes

Hay 62 fotos, 30 actas y 30 voces, pero NO existen vínculos documentados entre estas tres colecciones. En la ETL preserva ruta, modalidad, SHA-256, crédito y la relación PDF–PNG. Los registros de las 32 fotografías con ficha tienen metadatos; para las otras 30 no inventes autor ni licencia. Evita exportar el OCR completo de actas SECOP.

In [ ]:
info=json.loads((BASE/"datos/manifiestos/fotos_catalogadas_32.json").read_text())
fichas={Path(x["input_path"]).name:x for x in info["registros"]}
catalogo=[]
for foto in TODAS_FOTOS:
    meta=fichas.get(foto.name,{})
    catalogo.append({"ruta":str(foto.relative_to(BASE)),"tipo":"foto",
        "sha256":sha256(foto),"credito":meta.get("credit") or "pendiente",
        "url":meta.get("source_url") or ""})
for pdf in ACTAS:
    pagina=PAGINAS/(pdf.stem+"_pagina1.png")
    catalogo.append({"ruta":str(pdf.relative_to(BASE)),"tipo":"acta",
        "sha256":sha256(pdf),"vista_png":str(pagina.relative_to(BASE)),"credito":"SECOP"})
for voz in voces:
    ruta=BASE/voz["fuente"]
    catalogo.append({"ruta":voz["fuente"],"tipo":"audio","sha256":sha256(ruta),
                     "vista_wav":voz["pcm16"],"credito":"PolyAI MINDS-14 CC BY 4.0"})
assert len(catalogo)==122
archivo=OUT/"catalogo_122.jsonl"
with archivo.open("w",encoding="utf-8") as f:
    for fila in catalogo: f.write(json.dumps(fila,ensure_ascii=False)+"\n")
pd.DataFrame(catalogo).to_parquet(OUT/"catalogo_122.parquet",index=False)
display(pd.DataFrame(catalogo).groupby("tipo").size())

In [ ]:
releido=[json.loads(x) for x in archivo.read_text().splitlines() if x.strip()]
parquet=pd.read_parquet(OUT/"catalogo_122.parquet")
assert len(releido)==len(parquet)==122
for fila in releido: assert sha256(BASE/fila["ruta"])==fila["sha256"]
print("OK: 122 fuentes, todos los hashes coinciden")

## Reto final de investigación

Cambiar N_ACTAS a 30, N_FOTOS a 62, N_VISION a 62 o N_ASR a 10; comparar tiempos y errores con tu muestra pequeña. Elegir un archivo en el visor y contrastarlo con el resultado. Entregar una tabla: pregunta | corpus | número de archivos | cambio en el código | resultado | evidencia | limitación.

**No es aprendizaje ejecutar todas las celdas sin entenderlas.** Demuestra cómo construiste y modificaste los bucles y cuándo elegiste OCR, búsqueda visual o ASR.

**Abrir visor privado** desde la terminal: streamlit run visor_fuentes.py --server.address 0.0.0.0 --server.port 8501